# CART 498 — Assignment 1: Surrealistic Collage
**The Swap**: a three-part photomontage series made with Hugging Face `diffusers` (Stable Diffusion XL).

Collage the way Höch and Ernst did it: one base plate, and a different fragment cut out and pasted on each time. SDXL generates the base plate (a headless Victorian dress form, the Surrealists' favourite mannequin) and each "head" as a separate fragment. Python then cuts each fragment out as an oval and pastes it onto the neck: the goldfish moves into the birdcage, the canary into the fishbowl, and eventually they both escape.

Along the way I tried plain text-to-image and inpainting. Text-to-image put the birdcage *next to* the figure, and inpainting a neck kept producing a human head. Generating the pieces separately and pasting them is what made the swap land, and it keeps the cut-paper edge that makes it a collage.

Run on Google Colab with a T4 GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLInpaintPipeline, UNet2DConditionModel

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
INPAINT_ID = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, variant="fp16", use_safetensors=True
).to("cuda")

# Dedicated SDXL inpainting UNet (trained to fill a masked hole with a new object).
# Text encoders, VAE and scheduler are shared with `pipe`. It waits on the CPU: the T4 can't hold two SDXL UNets at once.
inpaint_unet = UNet2DConditionModel.from_pretrained(
    INPAINT_ID, subfolder="unet", torch_dtype=torch.float16, variant="fp16"
)
inpaint = StableDiffusionXLInpaintPipeline(**{**pipe.components, "unet": inpaint_unet})

In [ ]:
# Style shared by every generation so the fragments look cut from the same stack of magazines.
STYLE = "Dada photomontage collage, Hannah Hoch style, cut paper edges, halftone photo fragments, sepia, faded red, cream paper"

# The base plate: the same for all three images.
BASE_PROMPT = ("a headless Victorian dress form mannequin in a black high-collar dress, centered, "
               "standing beside a small table with a teacup, plain empty background above the collar, " + STYLE)

# The swap: the only fragment that changes, pasted where the head should be.
SWAPS = {
    1: "an antique brass birdcage with an orange goldfish swimming inside it",
    2: "a round glass fishbowl full of water with a grumpy yellow canary sitting inside it",
    3: "an empty antique brass birdcage with its little door hanging open, a yellow canary and an orange goldfish perched on top",
}

NEGATIVE = ("face, human head, eyes, portrait of a woman, photorealistic, 3d render, color photograph, "
            "modern, cartoon, anime, text, watermark, deformed, blurry")

tok = pipe.tokenizer
print("base", len(tok(BASE_PROMPT).input_ids), "tokens")
for i, s in SWAPS.items():
    print(i, len(tok(s + ", " + STYLE).input_ids), "tokens")

In [ ]:
# Seed sweep for the base plate: pick a seed where the dress form stands centered with room above the collar.
from IPython.display import display
from diffusers.utils import make_image_grid

CANDIDATES = [7, 42, 498, 1931, 2026, 4242]
previews = [
    pipe(prompt=BASE_PROMPT, negative_prompt=NEGATIVE, num_inference_steps=20, guidance_scale=7.5,
         width=768, height=768, generator=torch.Generator("cuda").manual_seed(s)).images[0].resize((256, 256))
    for s in CANDIDATES
]
print("seeds", CANDIDATES)
display(make_image_grid(previews, rows=1, cols=len(CANDIDATES)))

In [ ]:
# Full-resolution base plate from the chosen seed
from PIL import ImageDraw

SEED = 498
base = pipe(prompt=BASE_PROMPT, negative_prompt=NEGATIVE, num_inference_steps=40, guidance_scale=7.5,
            width=1024, height=1024, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]

# Preview with a 128px grid to locate the neck
guide = base.copy()
d = ImageDraw.Draw(guide)
for x in range(0, 1024, 128):
    d.line([(x, 0), (x, 1024)], fill="red", width=2)
    d.line([(0, x), (1024, x)], fill="red", width=2)
display(guide.resize((512, 512)))

In [ ]:
# Collage the Höch way: every piece is generated, then cut out and pasted.
import os
from PIL import Image as PILImage, ImageDraw, ImageFilter
from IPython.display import Image
from diffusers.utils import make_image_grid

# 1) Generate each "head" as its own fragment (text-to-image, same seed for all three).
fragments = {}
for i, swap in SWAPS.items():
    fragments[i] = pipe(
        prompt=swap + ", single object centered on plain cream paper, " + STYLE,
        negative_prompt=NEGATIVE, num_inference_steps=40, guidance_scale=7.5,
        width=1024, height=1024, generator=torch.Generator("cuda").manual_seed(SEED),
    ).images[0]
display(make_image_grid([f.resize((256, 256)) for f in fragments.values()], rows=1, cols=3))

# 2) Give the dress form headroom: slide the base plate down and outpaint the strip above it once,
#    so all three collages share exactly the same background. (Swap UNets: the T4 can't hold both.)
pipe.unet.to("cpu")
inpaint_unet.to("cuda")
torch.cuda.empty_cache()

SHIFT = 160
shifted = PILImage.new("RGB", base.size, (225, 212, 190))
shifted.paste(base, (0, SHIFT))
top_mask = PILImage.new("L", base.size, 0)
ImageDraw.Draw(top_mask).rectangle((0, 0, 1024, SHIFT + 24), fill=255)
plate = inpaint(
    prompt="plain empty studio wall, " + STYLE, negative_prompt=NEGATIVE, image=shifted, mask_image=top_mask,
    strength=0.99, num_inference_steps=30, guidance_scale=7.5, width=1024, height=1024,
    generator=torch.Generator("cuda").manual_seed(SEED),
).images[0]

# 3) Cut each fragment out as an oval, give it a paper border and a shadow, and paste it on the neck.
HEAD_BOX = (305, 8, 605, 338)  # (left, top, right, bottom): sits on the collar, read off the grid + SHIFT
w, h = HEAD_BOX[2] - HEAD_BOX[0], HEAD_BOX[3] - HEAD_BOX[1]
BORDER = 10

def oval(size):
    m = PILImage.new("L", size, 0)
    ImageDraw.Draw(m).ellipse((0, 0, size[0] - 1, size[1] - 1), fill=255)
    return m

os.makedirs("A1", exist_ok=True)
results = []
for i, frag in fragments.items():
    # keep the middle of the fragment, where the object is
    crop = frag.crop((140, 90, 884, 934)).resize((w, h), PILImage.LANCZOS)
    collage = plate.copy()
    shadow = PILImage.new("RGB", (w + 2 * BORDER, h + 2 * BORDER), (40, 30, 20))
    collage.paste(shadow, (HEAD_BOX[0] - BORDER + 8, HEAD_BOX[1] - BORDER + 8),
                  oval(shadow.size).filter(ImageFilter.GaussianBlur(8)))
    paper = PILImage.new("RGB", shadow.size, (238, 229, 210))
    collage.paste(paper, (HEAD_BOX[0] - BORDER, HEAD_BOX[1] - BORDER), oval(paper.size))
    collage.paste(crop, HEAD_BOX[:2], oval((w, h)))
    results.append(collage)

    collage.save(f"A1/surreal_collage_{i}.jpg", quality=92)
    with open(f"A1/surreal_collage_{i}.txt", "w") as f:
        f.write(f"Base plate (seed {SEED}): {BASE_PROMPT}\n\n"
                f"Head fragment {i} (seed {SEED}, cut out as an oval and pasted on the neck): "
                f"{SWAPS[i]}, single object centered on plain cream paper, {STYLE}\n")

display(make_image_grid([r.resize((512, 512)) for r in results], rows=1, cols=3))
for i in SWAPS:
    display(Image(filename=f"A1/surreal_collage_{i}.jpg", width=512))